In [0]:
from pyspark.sql import functions as F
import re

CATALOG = "healthcare_analytics"
BRONZE = "bronze"
SILVER = "silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER}")


def normalize_columns(df):
    """
    Standardize source column names to snake_case.
    """
    for old_name in df.columns:
        new_name = re.sub(r'[^a-zA-Z0-9]+', '_', old_name).strip('_').lower()
        df = df.withColumnRenamed(old_name, new_name)
    return df


def rename_if_exists(df, old_name, new_name):
    if old_name in df.columns and old_name != new_name:
        df = df.withColumnRenamed(old_name, new_name)
    return df


def cast_double_if_exists(df, column_name):
    if column_name in df.columns:
        df = df.withColumn(
            column_name,
            F.col(column_name).cast("double")
        )
    return df


def cast_date_if_exists(df, column_name):
    if column_name in df.columns:
        df = df.withColumn(
            column_name,
            F.to_date(F.col(column_name))
        )
    return df


def cast_double_if_exists(df, column_name):
    if column_name in df.columns:
        df = df.withColumn(
            column_name,
            F.expr(f"try_cast(`{column_name}` AS DOUBLE)")
        )
    return df


def bronze_table(name):
    return normalize_columns(
        spark.table(f"{CATALOG}.{BRONZE}.{name}_raw")
    )


def write_silver(df, table_name, keys=None):
    """
    Standard Silver writer with lineage timestamp and optional deduplication.
    """
    if keys:
        valid_keys = [k for k in keys if k in df.columns]
        if valid_keys:
            df = df.dropDuplicates(valid_keys)

    df = df.withColumn(
        "_silver_processed_at",
        F.current_timestamp()
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{CATALOG}.{SILVER}.{table_name}")
    )

    print(
        f"Created {CATALOG}.{SILVER}.{table_name}"
    )

In [0]:
# ============================================================
# PATIENTS
# ============================================================

patients = bronze_table("patients")

patients = rename_if_exists(patients, "id", "patient_id")
patients = rename_if_exists(patients, "birthdate", "birth_date")
patients = rename_if_exists(patients, "deathdate", "death_date")

patients = cast_date_if_exists(patients, "birth_date")
patients = cast_date_if_exists(patients, "death_date")

if "birth_date" in patients.columns:
    patients = patients.withColumn(
        "age",
        F.floor(
            F.months_between(
                F.current_date(),
                F.col("birth_date")
            ) / 12
        )
    )

    patients = patients.withColumn(
        "age_band",
        F.when(F.col("age") < 18, "0-17")
        .when(F.col("age") < 35, "18-34")
        .when(F.col("age") < 50, "35-49")
        .when(F.col("age") < 65, "50-64")
        .when(F.col("age") < 80, "65-79")
        .otherwise("80+")
    )

if "death_date" in patients.columns:
    patients = patients.withColumn(
        "is_deceased",
        F.col("death_date").isNotNull()
    )

name_columns = [
    c for c in ["first", "middle", "last"]
    if c in patients.columns
]

if name_columns:
    patients = patients.withColumn(
        "patient_name",
        F.concat_ws(
            " ",
            *[F.col(c) for c in name_columns]
        )
    )

write_silver(
    patients,
    "patients",
    ["patient_id"]
)


# ============================================================
# ENCOUNTERS
# ============================================================

encounters = bronze_table("encounters")

rename_map = {
    "id": "encounter_id",
    "patient": "patient_id",
    "organization": "organization_id",
    "provider": "provider_id",
    "payer": "payer_id",
    "start": "encounter_start",
    "stop": "encounter_end"
}

for old, new in rename_map.items():
    encounters = rename_if_exists(encounters, old, new)

encounters = cast_timestamp_if_exists(
    encounters,
    "encounter_start"
)

encounters = cast_timestamp_if_exists(
    encounters,
    "encounter_end"
)

for c in [
    "base_encounter_cost",
    "total_claim_cost",
    "payer_coverage"
]:
    encounters = cast_double_if_exists(encounters, c)

if (
    "encounter_start" in encounters.columns and
    "encounter_end" in encounters.columns
):
    encounters = encounters.withColumn(
        "encounter_duration_hours",
        (
            F.unix_timestamp("encounter_end")
            - F.unix_timestamp("encounter_start")
        ) / 3600
    )

if (
    "total_claim_cost" in encounters.columns and
    "payer_coverage" in encounters.columns
):
    encounters = encounters.withColumn(
        "patient_responsibility",
        F.greatest(
            F.col("total_claim_cost")
            - F.col("payer_coverage"),
            F.lit(0.0)
        )
    )

if "encounterclass" in encounters.columns:
    encounters = encounters.withColumnRenamed(
        "encounterclass",
        "encounter_class"
    )

if "encounter_class" in encounters.columns:
    encounters = encounters.withColumn(
        "is_inpatient",
        F.lower(F.col("encounter_class")) == "inpatient"
    )

if "encounter_start" in encounters.columns:
    encounters = (
        encounters
        .withColumn(
            "encounter_year",
            F.year("encounter_start")
        )
        .withColumn(
            "encounter_month",
            F.month("encounter_start")
        )
    )

write_silver(
    encounters,
    "encounters",
    ["encounter_id"]
)


# ============================================================
# ORGANIZATIONS / FACILITIES
# ============================================================

organizations = bronze_table("organizations")

organizations = rename_if_exists(
    organizations,
    "id",
    "organization_id"
)

organizations = rename_if_exists(
    organizations,
    "name",
    "organization_name"
)

write_silver(
    organizations,
    "organizations",
    ["organization_id"]
)


# ============================================================
# PROVIDERS
# ============================================================

providers = bronze_table("providers")

providers = rename_if_exists(
    providers,
    "id",
    "provider_id"
)

providers = rename_if_exists(
    providers,
    "organization",
    "organization_id"
)

providers = rename_if_exists(
    providers,
    "name",
    "provider_name"
)

write_silver(
    providers,
    "providers",
    ["provider_id"]
)


# ============================================================
# PAYERS
# ============================================================

payers = bronze_table("payers")

payers = rename_if_exists(
    payers,
    "id",
    "payer_id"
)

payers = rename_if_exists(
    payers,
    "name",
    "payer_name"
)

for c in payers.columns:
    if (
        "amount" in c
        or "revenue" in c
        or "coverage" in c
    ):
        payers = cast_double_if_exists(payers, c)

write_silver(
    payers,
    "payers",
    ["payer_id"]
)


# ============================================================
# CONDITIONS / DIAGNOSES
# ============================================================

conditions = bronze_table("conditions")

conditions = rename_if_exists(
    conditions,
    "patient",
    "patient_id"
)

conditions = rename_if_exists(
    conditions,
    "encounter",
    "encounter_id"
)

conditions = rename_if_exists(
    conditions,
    "start",
    "condition_start_date"
)

conditions = rename_if_exists(
    conditions,
    "stop",
    "condition_end_date"
)

conditions = cast_date_if_exists(
    conditions,
    "condition_start_date"
)

conditions = cast_date_if_exists(
    conditions,
    "condition_end_date"
)

if "condition_end_date" in conditions.columns:
    conditions = conditions.withColumn(
        "is_active_condition",
        F.col("condition_end_date").isNull()
    )

write_silver(
    conditions,
    "conditions"
)


# ============================================================
# PROCEDURES
# ============================================================

procedures = bronze_table("procedures")

procedures = rename_if_exists(
    procedures,
    "patient",
    "patient_id"
)

procedures = rename_if_exists(
    procedures,
    "encounter",
    "encounter_id"
)

procedures = rename_if_exists(
    procedures,
    "start",
    "procedure_start"
)

procedures = rename_if_exists(
    procedures,
    "stop",
    "procedure_end"
)

procedures = cast_timestamp_if_exists(
    procedures,
    "procedure_start"
)

procedures = cast_timestamp_if_exists(
    procedures,
    "procedure_end"
)

procedures = cast_double_if_exists(
    procedures,
    "base_cost"
)

if (
    "procedure_start" in procedures.columns and
    "procedure_end" in procedures.columns
):
    procedures = procedures.withColumn(
        "procedure_duration_minutes",
        (
            F.unix_timestamp("procedure_end")
            - F.unix_timestamp("procedure_start")
        ) / 60
    )

write_silver(
    procedures,
    "procedures"
)


# ============================================================
# MEDICATIONS
# ============================================================

medications = bronze_table("medications")

medications = rename_if_exists(
    medications,
    "patient",
    "patient_id"
)

medications = rename_if_exists(
    medications,
    "encounter",
    "encounter_id"
)

medications = rename_if_exists(
    medications,
    "payer",
    "payer_id"
)

medications = rename_if_exists(
    medications,
    "start",
    "medication_start_date"
)

medications = rename_if_exists(
    medications,
    "stop",
    "medication_end_date"
)

medications = cast_date_if_exists(
    medications,
    "medication_start_date"
)

medications = cast_date_if_exists(
    medications,
    "medication_end_date"
)

for c in [
    "base_cost",
    "payer_coverage",
    "totalcost"
]:
    medications = cast_double_if_exists(
        medications,
        c
    )

medications = rename_if_exists(
    medications,
    "totalcost",
    "total_cost"
)

if "medication_end_date" in medications.columns:
    medications = medications.withColumn(
        "is_active_medication",
        F.col("medication_end_date").isNull()
    )

write_silver(
    medications,
    "medications"
)


# ============================================================
# OBSERVATIONS / LABS / VITALS
# ============================================================

observations = bronze_table("observations")

observations = rename_if_exists(
    observations,
    "patient",
    "patient_id"
)

observations = rename_if_exists(
    observations,
    "encounter",
    "encounter_id"
)

observations = rename_if_exists(
    observations,
    "date",
    "observation_timestamp"
)

observations = cast_timestamp_if_exists(
    observations,
    "observation_timestamp"
)

if "value" in observations.columns:
    observations = observations.withColumn(
        "value_numeric",
        F.expr("try_cast(`value` AS DOUBLE)")
    )

    observations = observations.withColumn(
        "value_type",
        F.when(
            F.col("value_numeric").isNotNull(),
            F.lit("numeric")
        ).otherwise(
            F.lit("text")
        )
    )

write_silver(
    observations,
    "observations"
)


# ============================================================
# CLAIMS
# ============================================================

claims = bronze_table("claims")

claims = rename_if_exists(
    claims,
    "id",
    "claim_id"
)

claims = rename_if_exists(
    claims,
    "patientid",
    "patient_id"
)

claims = rename_if_exists(
    claims,
    "providerid",
    "provider_id"
)

claims = rename_if_exists(
    claims,
    "primarypatientinsuranceid",
    "primary_payer_id"
)

claims = rename_if_exists(
    claims,
    "secondarypatientinsuranceid",
    "secondary_payer_id"
)

for c in list(claims.columns):
    if (
        c.endswith("date")
        or c in [
            "servicedate",
            "currentillnessdate"
        ]
    ):
        claims = cast_date_if_exists(
            claims,
            c
        )

for c in list(claims.columns):
    if (
        "amount" in c
        or "cost" in c
        or "coverage" in c
        or "outstanding" in c
    ):
        claims = cast_double_if_exists(
            claims,
            c
        )

write_silver(
    claims,
    "claims",
    ["claim_id"]
)


# ============================================================
# CLAIM TRANSACTIONS
# ============================================================

claim_tx = bronze_table(
    "claims_transactions"
)

claim_tx = rename_if_exists(
    claim_tx,
    "patientid",
    "patient_id"
)

claim_tx = rename_if_exists(
    claim_tx,
    "claimid",
    "claim_id"
)

for c in list(claim_tx.columns):
    if (
        "amount" in c
        or "cost" in c
        or "payment" in c
    ):
        claim_tx = cast_double_if_exists(
            claim_tx,
            c
        )

write_silver(
    claim_tx,
    "claims_transactions"
)

Created healthcare_analytics.silver.patients
Created healthcare_analytics.silver.encounters
Created healthcare_analytics.silver.organizations
Created healthcare_analytics.silver.providers
Created healthcare_analytics.silver.payers
Created healthcare_analytics.silver.conditions
Created healthcare_analytics.silver.procedures
Created healthcare_analytics.silver.medications
Created healthcare_analytics.silver.observations
Created healthcare_analytics.silver.claims
Created healthcare_analytics.silver.claims_transactions


In [0]:
enhanced_tables = {
    "patients",
    "encounters",
    "organizations",
    "providers",
    "payers",
    "conditions",
    "procedures",
    "medications",
    "observations",
    "claims",
    "claims_transactions"
}

inventory = spark.table(
    "healthcare_analytics.bronze.bronze_file_inventory"
)

source_files = [
    r["source_file"]
    for r in inventory.select("source_file").collect()
]

for source_file in source_files:

    table_name = (
        source_file
        .lower()
        .replace(".csv", "")
    )

    if table_name in enhanced_tables:
        continue

    df = bronze_table(table_name)

    # Standard foreign-key naming
    df = rename_if_exists(
        df,
        "patient",
        "patient_id"
    )

    df = rename_if_exists(
        df,
        "encounter",
        "encounter_id"
    )

    # Common date/timestamp normalization
    for c in list(df.columns):

        if c in ["start", "stop"]:
            df = cast_timestamp_if_exists(
                df,
                c
            )

        elif c == "date":
            df = cast_timestamp_if_exists(
                df,
                c
            )

    write_silver(
        df,
        table_name
    )

print("Remaining Silver tables completed.")

Created healthcare_analytics.silver.allergies
Created healthcare_analytics.silver.careplans
Created healthcare_analytics.silver.devices
Created healthcare_analytics.silver.imaging_studies
Created healthcare_analytics.silver.immunizations
Created healthcare_analytics.silver.payer_transitions
Created healthcare_analytics.silver.supplies
Remaining Silver tables completed.


In [0]:
silver_tables = spark.sql("""
SHOW TABLES IN healthcare_analytics.silver
""")

display(silver_tables)

print(
    "Silver table count:",
    silver_tables.count()
)

database,tableName,isTemporary
silver,allergies,false
silver,careplans,false
silver,claims,false
silver,claims_transactions,false
silver,conditions,false
silver,devices,false
silver,encounters,false
silver,imaging_studies,false
silver,immunizations,false
silver,medications,false


Silver table count: 18


In [0]:
validation = []

for row in silver_tables.collect():

    table_name = row["tableName"]

    df = spark.table(
        f"healthcare_analytics.silver.{table_name}"
    )

    validation.append(
        (
            table_name,
            df.count(),
            len(df.columns)
        )
    )

validation_df = spark.createDataFrame(
    validation,
    [
        "table_name",
        "row_count",
        "column_count"
    ]
)

display(
    validation_df
    .orderBy(
        F.desc("row_count")
    )
)

table_name,row_count,column_count
claims_transactions,1005406,36
observations,770515,14
procedures,172791,14
imaging_studies,136217,16
claims,114186,34
encounters,63795,23
medications,50391,17
payer_transitions,41730,11
conditions,38675,11
supplies,28283,9
